# ViFinQA Hybrid Search RAG - Kaggle End-to-End Pipeline

This notebook guides you through the full execution of the RAG pipeline on Kaggle. It covers:
1. **Environment Setup**: Installing dependencies.
2. **ETL (Phase 1)**: Extracting financial tables from HTML files into structured CSVs.
3. **Indexing (Phase 2)**: Encoding metadata & building Qdrant local vector DB + BM25 index.
4. **Archiving/Packaging**: Creating a zip package of `rag_module/` (including indexes) to download or update your Kaggle Dataset.
5. **Evaluation**: Running retrieval validation against test questions.

In [ ]:
# ==========================================
# 1. Environment Setup & Constants
# ==========================================
!pip install -q sentence-transformers qdrant-client rank-bm25 pandas tqdm beautifulsoup4 lxml

from pathlib import Path
import os
import shutil

# --- PATH CONFIGURATION ---
# Adjust these according to your Kaggle input dataset names
INPUT_DATASET_DIR = Path("/kaggle/input/vifinqa-dataset") # Replace with your input dataset name
STATEMENTS_DIR = INPUT_DATASET_DIR / "financial_statements"
QUESTIONS_FILE = INPUT_DATASET_DIR / "questions/questions.jsonl"

# Output workspace directories (Kaggle working directory is writable)
PROCESSED_DIR = Path("/kaggle/working/processed_data")
QDRANT_DB_PATH = Path("/kaggle/working/qdrant_local_db")
BM25_PATH = Path("/kaggle/working/bm25_index.pkl")

# Code files location (assuming uploaded/cloned workspace structure)
CODE_STOCK_CSV = Path("rag_module/code_stock.csv")

print("Paths configured.")

## 2. Phase 1 - ETL (Extract HTML Tables to CSV)
This reads all `*_extracted.txt` documents under the raw financial statements folder, extracts their HTML tables, cleans them, appends company metadata, and writes them to individual CSV files in `/kaggle/working/processed_data/`.

In [ ]:
# Run ETL Phase
!python rag_module/data_pipeline.py \
    --statements-dir {STATEMENTS_DIR} \
    --processed-dir {PROCESSED_DIR} \
    --code-stock-csv {CODE_STOCK_CSV}

## 3. Phase 2 - Indexing (Build Vector DB & BM25)
This step reads the generated CSVs, encodes their contextual descriptions (Rich Content Strings), and updates:
- **Qdrant Vector DB** (Dense Retrieval)
- **BM25 Index** (Sparse Retrieval)

We pass `--skip-etl` so we don't repeat the HTML parsing, and `--run-indexing` to enable indexing.

In [ ]:
# Run Indexing Phase
!python rag_module/data_pipeline.py \
    --processed-dir {PROCESSED_DIR} \
    --qdrant-db-path {QDRANT_DB_PATH} \
    --bm25-path {BM25_PATH} \
    --code-stock-csv {CODE_STOCK_CSV} \
    --skip-etl \
    --run-indexing

## 4. Save and Package for Dataset Upload
Since the `/kaggle/working` folder is wiped out when the session ends, we copy the generated index databases into our `rag_module` directory, then package the entire folder as a ZIP file.

You can download `vifinqa-rag-module.zip` from your Kaggle notebook outputs and upload it as a new Dataset (e.g. `vifinqa-rag-module`), making it instantly importable in any of your inference notebooks!

In [ ]:
# Copy generated indexes to rag_module/ for packaging
dest_qdrant = Path("rag_module/qdrant_local_db")
dest_bm25 = Path("rag_module/bm25_index.pkl")

# Clear existing local indexes if they exist
if dest_qdrant.exists():
    shutil.rmtree(dest_qdrant)
if dest_bm25.exists():
    dest_bm25.unlink()

# Copy newly created databases
if QDRANT_DB_PATH.exists():
    shutil.copytree(QDRANT_DB_PATH, dest_qdrant)
    print("Copied Qdrant DB to rag_module/qdrant_local_db")

if BM25_PATH.exists():
    shutil.copy(BM25_PATH, dest_bm25)
    print("Copied BM25 Index to rag_module/bm25_index.pkl")

# Zip the rag_module directory
shutil.make_archive("/kaggle/working/vifinqa-rag-module", "zip", root_dir=".", base_dir="rag_module")
print("Successfully generated /kaggle/working/vifinqa-rag-module.zip!")

## 5. Test & Evaluate Retrieval
Finally, let's run the evaluation script with the newly built local indexes to verify that our hybrid search system returns valid tables.

In [ ]:
# Run evaluation script with 5 sample questions
!python rag_module/eval_retrieval.py \
    --questions {QUESTIONS_FILE} \
    --code-stock-csv {CODE_STOCK_CSV} \
    --num-questions 5 \
    --show-data